# Capstone — mirrors your deployed research paper

## 1. Question

**Core Research Question:**
*On this dataset, how well does a model trained on search, engagement, and content signals rank records carrying a recent historical search-impression decline label, compared with a hand-written scoring rule, when tested on clients not used for training? This is a retrospective comparison, not a test of advance warning.*

**Decision & Operational Context:**
- **Possible use:** Help an editor decide which records to inspect first; the ranking does not determine whether a page should be changed.
- **Actor:** Content strategists and digital editors.
- **Action:** An editor reviews the page and its context before deciding whether to make any change.
- **Limit:** This analysis measures agreement with a historical label. It does not measure whether a review or refresh improves traffic.

In [1]:
RESEARCH_QUESTION = "How well does a model rank records carrying a historical search-impression decline label, compared with a hand-written rule, on clients not used for training?"
TARGET_METRIC = "Share with historical decline label among top 50 (test clients)"
PRIMARY_ACTOR = "Content Operations & Editorial Team"

print(f"Research Question: {RESEARCH_QUESTION}")
print(f"Target Benchmark Metric: {TARGET_METRIC}")
print(f"Decision Consumer: {PRIMARY_ACTOR}")

Research Question: How well does a model rank records carrying a historical search-impression decline label, compared with a hand-written rule, on clients not used for training?
Target Benchmark Metric: Share with historical decline label among top 50 (test clients)
Decision Consumer: Content Operations & Editorial Team


## 2. Data

**Data Sources & Provenance:**
- **Dataset:** An anonymized starter snapshot (`data/raw/content_refresh_anonymized.csv`) with 30,000 content records and 32 pseudonymized client groups.
- **Inputs:** Search, engagement, and content information from the snapshot; the model does not use raw page text.
- **Excluded and restricted fields:**
  - `trend_direction` and `trend_pct` define or describe the label and are not model inputs.
  - `*_last_30d` fields are excluded. However, the model's 90-day inputs overlap the two 30-day periods used to create the label, so this is not an advance-warning test.
  - Client IDs are used only to keep each client in either training or testing; content IDs are not model inputs.
  - The published paper contains aggregate results only, not row-level records or pseudonymous IDs.

In [2]:
import os
import pandas as pd
import numpy as np

data_path = "../../data/raw/content_refresh_anonymized.csv" if os.path.exists("../../data/raw/content_refresh_anonymized.csv") else "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print(f"Verified Scope: {len(df):,} total pages across {df['client_id'].nunique()} distinct clients.")
print(f"Median Content Age: {df['content_age_days'].median():.0f} days.")
print(f"Rate scaling audit: Median CTR = {df['ctr'].median():.2f}% (confirmed x100 scale).")

Verified Scope: 30,000 total pages across 32 distinct clients.
Median Content Age: 236 days.
Rate scaling audit: Median CTR = 0.07% (confirmed x100 scale).


## 3. Methodology

**System Architecture & Assumptions:**
1. **Target:** A record is labelled as a decline when the latest 30-day impressions are more than 20% below the preceding 30 days. This describes a change that has already happened.
2. **Transparent Heuristic Baseline:**
   $$\text{Score}_{\text{baseline}} = 100 \times (0.40 \cdot S_{\text{visibility}} + 0.30 \cdot S_{\text{freshness}} + 0.25 \cdot S_{\text{position}})$$
3. **Trained model:** A Random Forest combines 100 decision trees. Each tree makes yes/no splits on the input information; their combined score ranks records by how likely they are to carry the historical decline label.
4. **Test design and limit:** One random split keeps entire clients separate (25 training clients, 7 test clients). The 90-day model inputs overlap the periods used to create the label, so the test measures retrospective ranking, not advance warning.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# Filter to mature content with search exposure
clean_df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy().reset_index(drop=True)
clean_df["is_declining_label"] = (clean_df["trend_direction"].str.lower() == "down").astype(int)

# Transform heavy-tailed signals
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    clean_df[f"log_{col}"] = np.log1p(clean_df[col].clip(lower=0))

feature_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]
cat_cols = ["competition_level", "content_type", "main_intent", "age_tier", "position_tier"]

X = pd.concat([clean_df[feature_cols].fillna(0), pd.get_dummies(clean_df[cat_cols].fillna("unknown"), drop_first=True)], axis=1)
y = clean_df["is_declining_label"].values

# Client-holdout split
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=clean_df["client_id"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Training on {len(X_train):,} pages ({clean_df.iloc[train_idx]['client_id'].nunique()} clients)")
print(f"Testing on  {len(X_test):,} pages ({clean_df.iloc[test_idx]['client_id'].nunique()} unseen clients)")

Training on 23,837 pages (25 clients)
Testing on  6,163 pages (7 unseen clients)


## 4. Results (vs baseline)

We compare both ranking methods on the same 6,163 records from clients absent from training. The top-20 and top-50 shares mean the fraction of records in each ranking that carry the historical decline label. ROC-AUC summarizes how often a decline-labelled record is ranked above a non-decline record across ranking cutoffs; 0.5 is chance-level ordering. This is one client split, not repeated testing.

In [4]:
from sklearn.metrics import roc_auc_score

def p_at_k(scores, y_true, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y_true)[order[:k]].mean()

# Baseline on test set
vis_t = np.log1p(clean_df.iloc[test_idx]["impressions_90d"].clip(lower=0)) / np.log1p(30000)
fresh_t = (clean_df.iloc[test_idx]["days_since_last_update"] / 180.0).clip(0, 1.0)
test_position = clean_df.iloc[test_idx]["avg_position"]
pos_t = np.where((test_position > 3) & (test_position <= 20), 1.0,
        np.where((test_position > 0) & (test_position <= 3), 0.5, 0.1))
base_scores = (0.40 * vis_t + 0.30 * fresh_t + 0.25 * pos_t) * 100.0

# Fit Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

# Summary benchmark
results_table = pd.DataFrame([
    {"Method": "Hand-written rule", "Top-20 decline-label share": p_at_k(base_scores, y_test, 20), "Top-50 decline-label share": p_at_k(base_scores, y_test, 50), "Overall ranking score (ROC-AUC)": roc_auc_score(y_test, base_scores)},
    {"Method": "Trained model", "Top-20 decline-label share": p_at_k(rf_probs, y_test, 20), "Top-50 decline-label share": p_at_k(rf_probs, y_test, 50), "Overall ranking score (ROC-AUC)": roc_auc_score(y_test, rf_probs)}
])

print("--- HEADLINE CAPSTONE RESULTS (Client-Holdout Evaluation) ---")
display(results_table.style.format({"Top-20 decline-label share": "{:.1%}", "Top-50 decline-label share": "{:.1%}", "Overall ranking score (ROC-AUC)": "{:.3f}"}))

p50_base = results_table.loc[0, "Top-50 decline-label share"]
p50_rf = results_table.loc[1, "Top-50 decline-label share"]
print(f"Test-set share with the decline label: {y_test.mean():.1%}")
print(f"On this single client-held-out test, {p50_rf:.1%} of the model's top 50 records and {p50_base:.1%} of the rule's top 50 records carried the historical decline label. This is a retrospective comparison, not evidence of advance warning or impact from editing pages.")

--- HEADLINE CAPSTONE RESULTS (Client-Holdout Evaluation) ---


,Method,Top-20 decline-label share,Top-50 decline-label share,Overall ranking score (ROC-AUC)
0,Hand-written rule,20.0%,34.0%,0.529
1,Trained model,45.0%,58.0%,0.609


Test-set share with the decline label: 51.1%
On this single client-held-out test, 58.0% of the model's top 50 records and 34.0% of the rule's top 50 records carried the historical decline label. This is a retrospective comparison, not evidence of advance warning or impact from editing pages.


## 5. Limitations

- The 90-day model inputs overlap the adjacent 30-day periods used to define the decline label. Results show retrospective alignment; they do not show that the model can identify future declines early.
- Only one random split of clients was tested. Results may differ on another split or in a later period.
- This comparison does not test whether editing a page changes its traffic. That would need a separate prospective experiment.
- The model uses numeric and categorical fields, not page text or external information that might explain traffic changes.

In [5]:
print("LIMITATIONS SUMMARY:")
print("1. Observational decision-support only (no causal claims without intervention trials).")
print("2. Google SERP feature shifts (AI overviews, zero-click answers) are unobserved.")
print("3. Retraining required when Google core updates trigger portfolio distribution shifts.")

LIMITATIONS SUMMARY:
1. Observational decision-support only (no causal claims without intervention trials).
2. Google SERP feature shifts (AI overviews, zero-click answers) are unobserved.
3. Retraining required when Google core updates trigger portfolio distribution shifts.


## 6. Ranked recommendations

For an exploratory queue only, we combine the model's decline-label score (70%) with the hand-written rule's score (30%). The model was trained on 25 clients, so queue scores for those clients use records seen during training. This full-dataset queue is not a test result or a validated recommendation.

In [6]:
# Compute full blended scores
vis_full = np.log1p(clean_df["impressions_90d"].clip(lower=0)) / np.log1p(30000)
fresh_full = (clean_df["days_since_last_update"] / 180.0).clip(0, 1.0)
position_full = clean_df["avg_position"]
pos_full = np.where((position_full > 3) & (position_full <= 20), 1.0,
           np.where((position_full > 0) & (position_full <= 3), 0.5, 0.1))
base_full = (0.40 * vis_full + 0.30 * fresh_full + 0.25 * pos_full) * 100.0

clean_df["model_prob"] = rf.predict_proba(X)[:, 1]
clean_df["final_refresh_score"] = 100.0 * (0.70 * clean_df["model_prob"] + 0.30 * (base_full / 100.0))

def get_action(r):
    if r["model_prob"] < 0.40 and r["final_refresh_score"] < 45: return "monitor"
    if 0 < r["word_count"] < 1200 and r["impressions_90d"] >= 250: return "expand_and_refresh"
    if r["impressions_90d"] >= 500 and 0 < r["avg_position"] <= 20 and r["ctr"] < 0.5: return "refresh_and_review_ctr"
    if r["sessions_90d"] >= 30 and (r["engagement_rate"] < 30 or r["scroll_rate"] < 30): return "refresh_and_review_engagement"
    return "refresh"

clean_df["recommended_action"] = clean_df.apply(get_action, axis=1)
capstone_queue = clean_df.sort_values("final_refresh_score", ascending=False).reset_index(drop=True)
capstone_queue["rank"] = capstone_queue.index + 1

action_summary = capstone_queue["recommended_action"].value_counts().rename_axis("Recommended action").reset_index(name="Pages")
print(f"Action queue contains {len(capstone_queue):,} eligible pages. Aggregate action counts:")
display(action_summary)

Action queue contains 30,000 eligible pages. Aggregate action counts:


,Recommended action,Pages
0,refresh,10877
1,refresh_and_review_ctr,9428
2,monitor,6428
3,refresh_and_review_engagement,3204
4,expand_and_refresh,63


## 7. Artifacts the paper embeds

We generate aggregate charts in `work/outputs/` for the deployed paper. Row-level queue records and pseudonymous IDs are not published.

In [7]:
import matplotlib.pyplot as plt
from pathlib import Path

repo_root = Path(data_path).resolve().parents[2]
output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Feature Importance Plot
fig, ax = plt.subplots(figsize=(9, 4))
top_feats = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(8)
top_feats.plot(kind="barh", color="#2ca02c", ax=ax)
display_names = {
    "days_with_impressions": "Days with impressions (90d)",
    "log_impressions_90d": "Search impressions (log, 90d)",
    "avg_position": "Average search position",
    "content_age_days": "Page age (days)",
    "word_count": "Word count",
    "char_count": "Character count",
    "ctr": "Click-through rate",
    "scroll_rate": "Scroll rate",
}
top_feats.index = [display_names.get(feature, feature) for feature in top_feats.index]
top_feats.plot(kind="barh", color="#2ca02c", ax=ax)
ax.set_title("Inputs most used for the historical decline label")
ax.set_xlabel("Relative tree-split importance (not causal impact)")
plt.tight_layout()
feat_chart_path = output_dir / "capstone_feature_importance.png"
fig.savefig(feat_chart_path, dpi=150)
plt.close(fig)
print(f"Generated and saved: {feat_chart_path}")

# 2. Precision Comparison Chart
from matplotlib.ticker import PercentFormatter

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(["Hand-written rule", "Trained model"], [p50_base, p50_rf], color=["#7f7f7f", "#1f77b4"], width=0.5)
ax.set_ylabel("Share carrying decline label (top 50)")
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("Historical decline labels among the top 50")
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.03, f"{yval:.0%}", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
prec_chart_path = output_dir / "capstone_precision_comparison.png"
fig.savefig(prec_chart_path, dpi=150)
plt.close(fig)
print(f"Generated and saved: {prec_chart_path}")

Generated and saved: C:\Users\divya\Desktop\FL_Machine_learning\work\outputs\capstone_feature_importance.png
Generated and saved: C:\Users\divya\Desktop\FL_Machine_learning\work\outputs\capstone_precision_comparison.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.